# Kenya demographic reconstruction and prioritisation

This notebook uses the same functions and pipeline as the command-line scripts. It examines the source, cleaning, exploratory data analysis, missingness, grouped splits, masking, model selection, prediction-error costs, and simulated prioritisation. All results are exploratory decision support, not official estimates.

Run from top to bottom with the project environment and the local HDX CSV in `data/raw/`. The full training pipeline may take considerable time and creates a new run directory; it never overwrites a previous run. For an existing complete run with matching input and configuration checksums, set `AML_NOTEBOOK_RUN_DIR` before starting Jupyter. Only use trusted model artifacts.

**Reading these results:** The cells were split after this run so each saved output has an adjacent interpretation. Repeated execution numbers identify outputs from the original combined execution. Numeric interpretations describe this specific run; rerun and revise them if the data or configuration changes.

In [1]:
import os
import sys
from pathlib import Path

from IPython.display import display

root = Path.cwd().resolve()
if not (root / 'run_pipeline.py').exists():
    root = root.parent
if not (root / 'run_pipeline.py').exists():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks directory')
sys.path.insert(0, str(root))
from src.analysis_workflow import (  # noqa: E402
    evaluate_selected,
    exploratory_figures,
    exploratory_tables,
    model_figures,
    obtain_run,
    read_config,
    run_outputs,
    source_and_profiles,
    source_tables,
)

config_path = root / 'config.yaml'
config = read_config(config_path)

c:\Users\HP\OneDrive\Documents\Dissertation\capstone_project_aml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Understanding the environment output

- The message about `IProgress` is a warning that an optional notebook progress-bar widget is unavailable. It is not a model result and, on its own, does not mean that training failed.
- The cell imports the libraries and shared project functions used below. The later completed pipeline and results confirm that this saved run continued past this warning.

In [1]:
config

{'seed': 42,
 'input': 'data/raw/hdx_hapi_population_global_non_hrp.csv',
 'output_root': 'data/processed/pipeline-runs',
 'prioritisation': {'budget': 10, 'generations': 200}}

### Understanding the configuration

- `seed: 42` controls the random choices used for splitting, masking and training so the same setup can be repeated.
- `input` points to the local population CSV; `output_root` is where separate pipeline runs are saved.
- The prioritisation settings use a budget of 10 simulated county units and 200 search generations. These units are not money or actual funding.

A fixed seed helps reproducibility, but it does not guarantee identical results after changing the data, code or software versions.

The configuration controls the fixed seed, raw input, output directory and simulation budget. Changing it changes the experiment; compare only runs with compatible inputs, splits and masks.

## Source and data understanding
The loader filters the local HDX HAPI file to Kenyan county records and checks that all 47 county codes exist. The field table describes the filtered records, not the global source.

In [2]:
source, profiles = source_and_profiles(config)
source_info = source_tables(source)
display(source_info['source_overview'])

,kenya_rows,columns,counties,duplicate_rows,reference_period_start,reference_period_end
0,3102,21,47,0,2019-01-01,2019-12-31


### Understanding the source overview

- `kenya_rows` is **3,102**: the number of records remaining after filtering the source to Kenya's county level. `columns` is **21**, the number of source fields.
- The records cover **47 counties**, with a reference period from **1 January to 31 December 2019**.
- `duplicate_rows: 0` means no complete source row is an exact duplicate. It does not by itself prove that every value is correct.

These figures describe the filtered Kenya extract, not the entire global dataset.

In [2]:
display(source_info['source_fields'])

,field,dtype,missing,distinct
0,location_code,str,0,1
1,has_hrp,str,0,1
2,in_gho,str,0,1
3,provider_admin1_name,str,0,47
4,provider_admin2_name,str,3102,0
5,admin1_code,str,0,47
6,admin1_name,str,0,47
7,admin2_code,str,3102,0
8,admin2_name,str,3102,0
9,admin_level,str,0,1


### Understanding the field table

Each row describes one source field. `dtype` is the type used when reading it, `missing` counts empty entries, and `distinct` counts different non-empty values.

- County codes and names each have 47 distinct values and no missing entries. `population` also has no missing entries.
- County-level data have no admin-2 values; this is expected because the extract is for admin level 1, not sub-counties.
- Some `min_age` and `max_age` values are absent in aggregate or open-ended age categories. This does not mean the population count itself is missing.

The loader initially reads values as text; cleaning checks and converts population values before analysis.

In [2]:
display(source_info['source_categories'])

,gender,age_range,rows
0,all,0-4,47
1,all,10-14,47
2,all,100+,47
3,all,15-19,47
4,all,20-24,47
...,...,...,...
61,m,80-84,47
62,m,85-89,47
63,m,90-94,47
64,m,95-99,47


### Understanding the demographic categories

The table groups records by `gender` and `age_range`; `rows` tells us how many records occur in each combination. There are **66 combinations**, and each has **47 rows**, matching one entry for each county.

The `all` category represents a reported aggregate. It overlaps with the separate age or sex categories, so adding it to those components would count the same people more than once. A balanced count of rows is useful but does not prove that the values are accurate.

In [2]:
display(source.head())

,location_code,has_hrp,in_gho,provider_admin1_name,provider_admin2_name,admin1_code,admin1_name,admin2_code,admin2_name,admin_level,...,age_range,min_age,max_age,population,reference_period_start,reference_period_end,dataset_hdx_id,resource_hdx_id,warning,error
0,KEN,N,Y,Mombasa,NaN,KE001,Mombasa,NaN,NaN,1,...,all,NaN,NaN,598046,2019-01-01,2019-12-31,26801642-7088-4149-9605-6ae282897937,d4fed43d-8abe-48d2-987a-35c101292af3,NaN,NaN
1,KEN,N,Y,Mombasa,NaN,KE001,Mombasa,NaN,NaN,1,...,all,NaN,NaN,610257,2019-01-01,2019-12-31,26801642-7088-4149-9605-6ae282897937,d4fed43d-8abe-48d2-987a-35c101292af3,NaN,NaN
2,KEN,N,Y,Mombasa,NaN,KE001,Mombasa,NaN,NaN,1,...,all,NaN,NaN,1208303,2019-01-01,2019-12-31,26801642-7088-4149-9605-6ae282897937,d4fed43d-8abe-48d2-987a-35c101292af3,NaN,NaN
3,KEN,N,Y,Mombasa,NaN,KE001,Mombasa,NaN,NaN,1,...,0-4,0,4,73178,2019-01-01,2019-12-31,26801642-7088-4149-9605-6ae282897937,d4fed43d-8abe-48d2-987a-35c101292af3,NaN,NaN
4,KEN,N,Y,Mombasa,NaN,KE001,Mombasa,NaN,NaN,1,...,5-9,5,9,65459,2019-01-01,2019-12-31,26801642-7088-4149-9605-6ae282897937,d4fed43d-8abe-48d2-987a-35c101292af3,NaN,NaN


### Understanding the raw rows

This preview shows the first five source records, all for Mombasa. In this **long format**, each row represents a county and a particular sex and age category; `population` is the reported count.

The `all` age rows are totals, while rows such as `0-4` describe individual age groups. Empty admin-2 fields reflect the county-level extract. A five-row preview helps us understand the structure, but cannot establish the quality of the whole dataset.

Review the reference period, field completeness, category counts and duplicates above before drawing conclusions. Source values are initially read as strings; the cleaner validates and converts population values. A single historical extract cannot establish trends or causal relationships.

## Cleaning, distributions and missingness
Cleaning creates one county profile while preserving reported aggregate totals and missing observations. The same analysis functions also produce CSV and HTML reports in every new pipeline run.

In [3]:
tables = exploratory_tables(profiles)
figures = exploratory_figures(tables)
display(tables['overview'])

,counties,profile_columns,atomic_cells,missing_atomic_cells,duplicate_county_codes
0,47,68,1974,0,0


### Understanding the cleaned dataset

Cleaning produces **47 rows**, one per county, and **68 columns** including identifiers and population fields. The 42 age-by-sex fields per county form **1,974 atomic cells** (47 times 42); `missing_atomic_cells: 0` means all are present. There are no duplicate county codes.

An atomic cell is one count for a particular sex and age band. Complete cells do not guarantee that they sum exactly to the separately reported totals; the consistency checks below examine that.

In [3]:
display(profiles.head())

,admin1_code,admin1_name,population_f_0_4,population_f_5_9,population_f_10_14,population_f_15_19,population_f_20_24,population_f_25_29,population_f_30_34,population_f_35_39,...,population_all_60_64,population_all_65_69,population_all_70_74,population_all_75_79,population_all_80_84,population_all_85_89,population_all_90_94,population_all_95_99,population_all_100_plus,population_all_all
0,KE001,Mombasa,73178,65459,58280,53189,75911,72801,60864,42035,...,16034,10243,6395,3045,1858,803,323,152,73,1208303
1,KE002,Kwale,64761,65043,62091,47688,39092,34388,30715,20499,...,16965,12853,9437,4616,3260,1604,700,306,259,866802
2,KE003,Kilifi,99912,102458,103193,85085,71704,60525,54385,32187,...,29594,21335,15301,8726,5336,3002,1158,667,363,1453762
3,KE004,Tana River,26006,25878,22971,16060,14218,11740,9491,6624,...,5602,3616,2904,1362,1003,484,244,104,87,315941
4,KE005,Lamu,9134,9241,8981,7229,6036,5083,4882,3560,...,2978,2197,1738,800,517,226,125,53,49,143916


### Understanding the cleaned profiles

The table is now in **wide format**: each row is a county and each population column represents a specific sex and age group or a reported total. For example, `population_f_0_4` is the female population aged 0 to 4; Mombasa's value is **73,178**.

The identifiers are used to track and join counties. These values come from the source after cleaning; they are not predictions made by the model.

In [3]:
display(tables['county_totals'])

,admin1_code,admin1_name,population_all_all
46,KE047,Nairobi,4396828
21,KE022,Kiambu,2417600
31,KE032,Nakuru,2162107
36,KE037,Kakamega,1867539
38,KE039,Bungoma,1670535
11,KE012,Meru,1545673
2,KE003,Kilifi,1453762
15,KE016,Machakos,1421898
44,KE045,Kisii,1266822
0,KE001,Mombasa,1208303


### Understanding reported county totals

The table ranks counties by `population_all_all`, the source's reported total across sexes and ages. Nairobi is highest at **4,396,828** and Lamu is lowest at **143,916** in this 2019 extract.

These values show a large difference in county size. They are reported totals, not the sum recalculated from the atomic cells, and the later checks show that those two figures may differ.

In [3]:
display(tables['atomic_distribution'])

,age_sex,count,mean,std,min,25%,50%,75%,max
0,population_f_0_4,47.0,63548.276596,41699.834775,9134.0,33273.5,56989.0,75281.5,260888.0
1,population_m_0_4,47.0,63964.765957,41950.481852,9604.0,34176.0,57159.0,74627.0,264099.0
2,population_f_5_9,47.0,65649.276596,37842.057232,9241.0,37147.0,64448.0,78965.5,217482.0
3,population_m_5_9,47.0,66318.106383,37584.689051,9505.0,38099.0,64527.0,77528.0,215230.0
4,population_f_10_14,47.0,66726.425532,36968.699694,8981.0,38906.0,63327.0,84213.5,193542.0
5,population_m_10_14,47.0,68292.765957,36014.210965,9492.0,40791.5,65946.0,85902.5,185008.0
6,population_f_15_19,47.0,55307.276596,33559.005284,7229.0,34028.5,52400.0,67875.5,192755.0
7,population_m_15_19,47.0,57154.553191,30324.911731,7923.0,35058.5,56100.0,67896.5,159098.0
8,population_f_20_24,47.0,49676.106383,47206.852872,6036.0,24309.0,41185.0,59456.0,313485.0
9,population_m_20_24,47.0,44950.851064,37940.534797,7134.0,23630.5,38235.0,51020.0,249534.0


### Understanding the distribution table

Each row summarizes one age-sex population column across the 47 counties:

- `count` is the number of observed counties; `mean` is their average and `std` measures how spread out the values are.
- `min` and `max` are the smallest and largest county counts. `25%`, `50%` and `75%` are the lower quartile, median and upper quartile.
- For females aged 0 to 4, the mean is about **63,548** and the standard deviation is about **41,700**, showing substantial variation between counties.

These statistics describe counts per county, not national totals or population-adjusted rates.

In [3]:
display(tables['demographics'])

,age_sex,count,mean,median,minimum,maximum
0,population_f_0_4,47,63548.276596,56989.0,9134,260888
1,population_m_0_4,47,63964.765957,57159.0,9604,264099
2,population_f_5_9,47,65649.276596,64448.0,9241,217482
3,population_m_5_9,47,66318.106383,64527.0,9505,215230
4,population_f_10_14,47,66726.425532,63327.0,8981,193542
5,population_m_10_14,47,68292.765957,65946.0,9492,185008
6,population_f_15_19,47,55307.276596,52400.0,7229,192755
7,population_m_15_19,47,57154.553191,56100.0,7923,159098
8,population_f_20_24,47,49676.106383,41185.0,6036,313485
9,population_m_20_24,47,44950.851064,38235.0,7134,249534


### Understanding the age-sex summary

This table lists, for each age-sex group, the number of observations and its mean, median, minimum and maximum across counties. The **median** is the middle county value when the 47 counts are ordered.

For ages 0 to 4, the median is **56,989 females** and **57,159 males**. Comparing the median with the mean and extremes helps reveal groups whose values vary widely between counties. Do not add these median values to obtain a national population.

In [3]:
figures['county_population'].show()

### Understanding the county population chart

The horizontal axis lists counties; the height of each bar on the vertical axis is the county's **reported population**. The tall Nairobi bar corresponds to **4,396,828**, while Lamu has **143,916**.

This makes differences in county size easy to see. A large absolute prediction error may have a different practical significance in a large county than in a small one; this chart does not itself measure model error.

In [3]:
figures['county_distribution'].show()

### Understanding the population histogram

The horizontal axis groups reported county populations into ranges, called bins. The vertical axis counts how many counties fall in each range. Each county contributes one observation, regardless of its population size.

The county median is **893,653**, while Nairobi's population is much larger. This shows why an average alone may hide differences between typical and unusually large counties. The chart describes counties, not the number of individual people in each age group.

In [3]:
figures['age_sex'].show()

### Understanding the age-sex chart

The horizontal axis lists age-sex fields and the vertical axis shows the **median count across counties** for each field. For example, the median female 0-to-4 count is **56,989**.

Younger age groups generally have higher median counts than older groups in this extract. These are separate county medians from one 2019 snapshot, not national totals, population percentages or evidence of change over time.

In [3]:
figures['dependency'].show()

### Understanding the dependency-ratio chart

Each bar represents a county. The vertical axis is `(children aged 0-14 + people aged 65 and above) / people aged 15-64`, calculated from observed age-sex counts.

Nairobi's ratio is about **0.46**, or roughly 46 children and older people per 100 working-age people. Mandera's is about **1.29**, or roughly 129 per 100. This demographic indicator is not a direct measure of poverty, employment, service demand or fairness.

The county chart uses reported totals; the age-sex table and chart summarize the 42 atomic cells. Large differences between counties or age groups can make absolute error and RMSE disproportionately sensitive to larger populations. These are counts, not rates adjusted for county population.

In [4]:
display(tables['missing_by_county'])

,admin1_code,admin1_name,missing_count,missing_fraction
0,KE001,Mombasa,0,0.0
1,KE002,Kwale,0,0.0
2,KE003,Kilifi,0,0.0
3,KE004,Tana River,0,0.0
4,KE005,Lamu,0,0.0
5,KE006,Taita Taveta,0,0.0
6,KE007,Garissa,0,0.0
7,KE008,Wajir,0,0.0
8,KE009,Mandera,0,0.0
9,KE010,Marsabit,0,0.0


### Understanding missingness by county

For each county, `missing_count` counts empty population fields and `missing_fraction` divides that count by the number of checked population fields. Every county has a count and fraction of **zero** in this cleaned extract.

This is a finding about the available 2019 data, not a guarantee that future data will be complete. The model experiments create missing values deliberately so reconstruction can be evaluated.

In [4]:
display(tables['missing_by_column'])

,column,missing_count,missing_fraction
0,population_f_0_4,0,0.0
1,population_f_5_9,0,0.0
2,population_f_10_14,0,0.0
3,population_f_15_19,0,0.0
4,population_f_20_24,0,0.0
...,...,...,...
61,population_all_85_89,0,0.0
62,population_all_90_94,0,0.0
63,population_all_95_99,0,0.0
64,population_all_100_plus,0,0.0


### Understanding missingness by column

Each row is a population field; `missing_count` counts counties without a value and `missing_fraction` is that count divided by 47. All **66 population fields** have zero missing values in the cleaned profiles.

This does not conflict with missing admin-2 or age-bound metadata in the raw-field table: those are different columns and are not population counts.

In [4]:
display(tables['consistency_summary'])

,check,status,count
0,age_all,inconsistent,47
1,age_f,consistent,2
2,age_f,inconsistent,45
3,age_m,consistent,2
4,age_m,inconsistent,45
5,gender_0-4,consistent,47
6,gender_10-14,consistent,47
7,gender_100+,consistent,47
8,gender_15-19,consistent,47
9,gender_20-24,consistent,47


### Understanding the consistency checks

These checks compare reported totals with their components. A `consistent` check adds up exactly; `inconsistent` means it does not; an `incomplete` check lacks a required value.

- There are **1,038 consistent** and **137 inconsistent** checks, with no incomplete checks.
- All 47 combined-sex age totals, 45 female totals and 45 male totals differ from the sum of their age components. The checked female-plus-male sums agree with reported totals for each age and overall.

These are discrepancies in the supplied source values. They should be investigated and reported, not automatically treated as model mistakes or silently corrected.

In [4]:
display(tables['consistency_checks'].query("status != 'consistent'"))

,admin1_code,admin1_name,check,status,difference
1034,KE001,Mombasa,age_f,inconsistent,12
1035,KE002,Kwale,age_f,inconsistent,6
1036,KE003,Kilifi,age_f,inconsistent,11
1037,KE004,Tana River,age_f,inconsistent,1
1038,KE005,Lamu,age_f,inconsistent,1
...,...,...,...,...,...
1170,KE043,Homa Bay,age_all,inconsistent,9
1171,KE044,Migori,age_all,inconsistent,17
1172,KE045,Kisii,age_all,inconsistent,22
1173,KE046,Nyamira,age_all,inconsistent,4


### Understanding the inconsistent rows

Each row identifies a county and the `check` that failed. `difference` is **reported total minus the sum of its age components**: a positive number means the reported total is larger.

For example, the female total for Mombasa is **12** higher than its age components, and Nairobi's combined-sex total is **143** higher. The table contains **137 rows**; it points to checks needing investigation but does not explain why the source values differ.

In [4]:
figures['atomic_missingness'].show()

### Understanding the missingness chart

The horizontal axis lists population fields and the vertical axis is the proportion of counties with a missing value in each field. All plotted fractions are **0**, consistent with the missingness tables above.

The chart uses the cleaned population fields. The model's missing values are later simulated by hiding known cells, which may not reproduce the reasons or patterns of missingness in future real datasets.

Missing values are not automatically zeros. Inconsistent source totals are reported rather than silently corrected; their differences also affect later consistency metrics. A complete source does not prove that simulated missingness matches real-world missingness.

## Reproducible pipeline: features, masks, training and saving
The shared pipeline creates disjoint county-grouped train, validation and test splits; simulates random, age-block and county-block masks; trains the tree and autoencoder candidates using grouped cross-validation; selects by validation MAE; saves model artifacts; and runs prioritisation on observed profiles. Test counties are not used in selection.

In [5]:
existing = os.environ.get('AML_NOTEBOOK_RUN_DIR')
run_dir = obtain_run(config_path, Path(existing) if existing else None)
outputs = run_outputs(run_dir)

Source totals: 137 inconsistent and 0 incomplete checks; see C:\Users\HP\OneDrive\Documents\Dissertation\capstone_project_aml\data\processed\pipeline-runs\notebook-20261120T152551396392Z\reports\consistency_checks.csv


### Understanding the pipeline warning

The warning records **137 inconsistent** and **0 incomplete** source-total checks and tells you where the detailed report was saved. The pipeline continues so those discrepancies remain visible for analysis; it does not claim that they were fixed.

The warning describes the raw data quality, not a failed model prediction. Check the report before treating the reported totals as interchangeable with sums of age-sex values.

In [5]:
display({'run_directory': str(run_dir), 'manifest': outputs['manifest']})

{'run_directory': 'C:\\Users\\HP\\OneDrive\\Documents\\Dissertation\\capstone_project_aml\\data\\processed\\pipeline-runs\\notebook-20261120T152551396392Z',
 'manifest': {'status': 'complete',
  'input_sha256': 'd113eeb87355d795b3f00dea57010fa2e3d723459ee647a20b4aba644608c048',
  'processed_sha256': 'a040134935fb992e176c44163e78565bff40518f5e7cd0d6e0ec91b9c2d1ffd1',
  'config_sha256': '6cf6508005151262763d641088d52a445312090a184d39fe09f99170511cc516',
  'config': {'seed': 42,
   'input': 'data/raw/hdx_hapi_population_global_non_hrp.csv',
   'output_root': 'data/processed/pipeline-runs',
   'prioritisation': {'budget': 10, 'generations': 200}},
  'counties': 47,
  'inconsistent_checks': 137,
  'incomplete_checks': 0,
  'optimisation_input': 'observed profiles',
  'test_evaluated': False}}

### Understanding the run manifest

The manifest records what happened in this pipeline run:

- `status: complete` means the pipeline reached its final step; `counties: 47` records the number of profiles.
- The SHA-256 values are fingerprints of the input, processed file and configuration. They help identify changed files, but do not record every code or software dependency version.
- The seed and settings describe the experiment; the counts record the source-total warnings.
- `optimisation_input` says prioritisation used **observed** profiles. `test_evaluated: false` means the pipeline did not score the reserved test split.

The run directory holds this run's separate reports and model files; it does not automatically replace the models used by the API.

In [5]:
display(outputs['experiments'])

,pattern,requested_rate,split,counties,observed_cells,masked_cells,actual_cell_fraction
0,random,0.1,train,33,1386,139,0.100289
1,random,0.1,validation,7,294,29,0.098639
2,random,0.1,test,7,294,29,0.098639
3,random,0.2,train,33,1386,277,0.199856
4,random,0.2,validation,7,294,59,0.200680
5,random,0.2,test,7,294,59,0.200680
6,random,0.4,train,33,1386,554,0.399711
7,random,0.4,validation,7,294,118,0.401361
8,random,0.4,test,7,294,118,0.401361
9,age-block,0.1,train,33,1386,138,0.099567


### Understanding the mask experiment table

The model is tested by hiding known values and asking it to reconstruct them. Each row describes one combination of mask pattern, requested rate and data split.

- `random` hides individual cells; `age-block` hides age groups; `county-block` hides whole county profiles.
- `train`, `validation` and `test` contain **33, 7 and 7 counties**, respectively. Counties stay in one split to reduce leakage.
- `observed_cells`, `masked_cells` and `actual_cell_fraction` show how much data was available and actually hidden. Rounding can make the actual fraction differ from the requested rate.

There are **27 rows**: three patterns times three rates times three splits. Preparing test masks does not mean the test set has been evaluated.

In [5]:
display(outputs['selection'])

{'selection_metric': 'unweighted mean validation MAE across the nine masking scenarios',
 'selected_family': 'random_forest',
 'mean_mae_by_family': {'random_forest': 5582.3155638175,
  'autoencoder': 5637.472715106525,
  'xgboost': 5779.736690966368},
 'input_sha256': 'a040134935fb992e176c44163e78565bff40518f5e7cd0d6e0ec91b9c2d1ffd1',
 'seed': 42,
 'artifact_files': ['random_10_random_forest.joblib',
  'random_20_random_forest.joblib',
  'random_40_random_forest.joblib',
  'age-block_10_random_forest.joblib',
  'age-block_20_random_forest.joblib',
  'age-block_40_random_forest.joblib',
  'county-block_10_random_forest.joblib',
  'county-block_20_random_forest.joblib',
  'county-block_40_random_forest.joblib'],
 'test_evaluated': False}

### Understanding model selection

The selected family is **random forest**. The selection rule averages validation MAE equally across all nine masking scenarios, rather than choosing the model with the best score in only one scenario. MAE is the average absolute error in persons per hidden cell; lower is better.

| Model family | Mean validation MAE |
|---|---:|
| Random forest | 5,582 |
| Autoencoder | 5,637 |
| XGBoost | 5,780 |

The roughly **55-person** gap between the first two is small relative to their errors and is based on a small validation sample; it does not establish a statistically significant advantage. `artifact_files` lists the nine scenario-specific saved models, and `test_evaluated: false` shows selection did not use test scores.

The manifest records checksums, seed and data-quality warnings. Each mask summary lists actual held-out cells, which may differ from the requested percentage. Selection averages the nine validation MAEs equally, and artifacts are in `models/selected` within this run. They are not automatically deployed. Reusing an existing run requires matching checksums, but these do not verify its original code revision or dependency versions.

## Model quality, generalisation and overfitting risk
Examine every scenario rather than only the average. MAE and RMSE are in persons per masked cell; RMSE emphasizes large errors. R-squared can be negative or undefined when targets are constant. Grouped CV uses training counties, while validation uses different counties.

In [6]:
display(outputs['comparison'])

,pattern,rate,model,split,train_counties_with_targets,train_targets,validation_targets,mask_sha256,cv_mae,best_params,...,rmse,r2,negative_cells,observed_cells,gender_checks,gender_inconsistent,gender_absolute_difference,age_checks,age_inconsistent,age_absolute_difference
0,random,0.1,random_forest,validation,32,139,29,8d1cedade7821807dd3abb5cd6295d8b21c4fd05835d33...,6646.393611,"{""max_features"": 1.0, ""min_samples_leaf"": 1, ""...",...,4903.387080,0.943414,0,294,154,26,85624.140000,21,21,63862.960000
1,random,0.1,xgboost,validation,32,139,29,8d1cedade7821807dd3abb5cd6295d8b21c4fd05835d33...,6155.111994,"{""learning_rate"": 0.1, ""max_depth"": 4, ""n_esti...",...,5549.974263,0.927507,0,294,154,26,97890.530975,21,21,80276.779999
2,random,0.2,random_forest,validation,33,277,59,8ed7947251eca65ae92af569d06ceda63edbcfb1309350...,8188.570349,"{""max_features"": 0.7, ""min_samples_leaf"": 1, ""...",...,7157.155366,0.923843,0,294,154,53,264410.250000,21,21,177205.850000
3,random,0.2,xgboost,validation,33,277,59,8ed7947251eca65ae92af569d06ceda63edbcfb1309350...,9585.824422,"{""learning_rate"": 0.1, ""max_depth"": 4, ""n_esti...",...,9586.251494,0.863376,1,294,154,53,352738.003708,21,21,328405.418625
4,random,0.4,random_forest,validation,33,554,118,b8add2e9936f6567efcdd544d1fc3a0fb99d1cfe93cbe8...,5750.136544,"{""max_features"": 0.7, ""min_samples_leaf"": 1, ""...",...,7582.777225,0.903164,0,294,154,92,493455.320000,21,21,409113.000000
5,random,0.4,xgboost,validation,33,554,118,b8add2e9936f6567efcdd544d1fc3a0fb99d1cfe93cbe8...,7538.307498,"{""learning_rate"": 0.1, ""max_depth"": 2, ""n_esti...",...,8386.772380,0.881540,37,294,154,92,710616.188484,21,21,529305.056206
6,age-block,0.1,random_forest,validation,29,138,30,90de7d3e814a883717c338d7fd554509e6488ac3e9af73...,6616.754203,"{""max_features"": 1.0, ""min_samples_leaf"": 1, ""...",...,10325.823583,0.800392,0,294,154,15,209939.100000,21,21,207984.080000
7,age-block,0.1,xgboost,validation,29,138,30,90de7d3e814a883717c338d7fd554509e6488ac3e9af73...,6679.067036,"{""learning_rate"": 0.1, ""max_depth"": 2, ""n_esti...",...,8429.235645,0.866983,0,294,154,15,176512.304199,21,21,172605.274902
8,age-block,0.2,random_forest,validation,33,278,58,7f6c872939030ed45bd7fc480d2a9f6480950633552002...,5676.489335,"{""max_features"": 1.0, ""min_samples_leaf"": 1, ""...",...,7903.137429,0.870473,0,294,154,29,241451.730000,21,21,183525.250000
9,age-block,0.2,xgboost,validation,33,278,58,7f6c872939030ed45bd7fc480d2a9f6480950633552002...,6248.026358,"{""learning_rate"": 0.1, ""max_depth"": 4, ""n_esti...",...,5269.854327,0.942409,0,294,154,29,194397.045792,21,21,164111.119492


### Understanding the candidate comparison table

Each row reports one model family for one mask pattern and rate, using the same validation scenario. The table includes:

- `train_counties_with_targets`, `train_targets` and `validation_targets`: how many counties or hidden values contributed to training or scoring.
- `cv_mae` and `best_params`: the grouped training cross-validation score and selected settings. `mask_sha256` identifies the validation mask used for a fair scenario comparison.
- `mae`, `rmse` and `r2`: errors and fit on hidden validation values. The remaining fields report negative predictions and agreement with separately reported totals, which themselves contain source discrepancies.

The forest wins **on average**, not everywhere: the autoencoder has lower MAE for age-block 10%, and XGBoost for age-block 20%. The autoencoder also receives aggregate training supervision not available to the trees, so the comparison is not perfectly like-for-like.

In [6]:
display(outputs['diagnostics'])

,split,pattern,rate,model,counties,targets,mask_sha256,cv_mae,mae,rmse,...,median_baseline_mae,median_baseline_rmse,median_baseline_r2,bias,mean_underestimate,mean_overestimate,underestimate_fraction,cost_equal,cost_under_double,cost_over_double
0,validation,random,0.1,random_forest,7,29,8d1cedade7821807dd3abb5cd6295d8b21c4fd05835d33...,6646.393611,2952.556552,4903.387080,...,5481.206897,9564.906195,0.784683,-209.262759,1580.909655,1371.646897,0.551724,2952.556552,4533.466207,4324.203448
1,validation,random,0.2,random_forest,7,59,8ed7947251eca65ae92af569d06ceda63edbcfb1309350...,8188.570349,4509.167288,7157.155366,...,7988.728814,12547.293476,0.765939,-2126.377458,3317.772373,1191.394915,0.440678,4509.167288,7826.939661,5700.562203
2,validation,random,0.4,random_forest,7,118,b8add2e9936f6567efcdd544d1fc3a0fb99d1cfe93cbe8...,5750.136544,4226.043729,7582.777225,...,7551.720339,13231.033603,0.705172,-2867.553220,3546.798475,679.245254,0.567797,4226.043729,7772.842203,4905.288983
3,validation,age-block,0.1,random_forest,7,30,90de7d3e814a883717c338d7fd554509e6488ac3e9af73...,6616.754203,7003.177333,10325.823583,...,5724.866667,9787.730040,0.820653,352.298667,3325.439333,3677.738000,0.366667,7003.177333,10328.616667,10680.915333
4,validation,age-block,0.2,random_forest,7,58,7f6c872939030ed45bd7fc480d2a9f6480950633552002...,5676.489335,4286.059138,7903.137429,...,4143.362069,8075.641058,0.864757,-2347.285690,3316.672414,969.386724,0.586207,4286.059138,7602.731552,5255.445862
5,validation,age-block,0.4,random_forest,7,118,788cc7dfdf44a53bac614efd4d96be9de53873545f2dcc...,4401.937676,5556.015593,9370.211462,...,6301.644068,10449.856751,0.812729,-4242.602034,4899.308814,656.706780,0.627119,5556.015593,10455.324407,6212.722373
6,validation,county-block,0.1,random_forest,7,42,15cd011e17274ac25a0d083b8a3de081f4ff5dad262b05...,26985.872734,7753.585350,11902.221727,...,9526.190476,15226.681671,0.704451,-4454.063707,6103.824529,1649.760822,0.404762,7753.585350,13857.409879,9403.346172
7,validation,county-block,0.2,random_forest,7,42,15cd011e17274ac25a0d083b8a3de081f4ff5dad262b05...,27433.949548,6993.888781,11648.536696,...,9526.190476,15226.681671,0.704451,6813.484366,90.202208,6903.686574,0.071429,6993.888781,7084.090989,13897.575355
8,validation,county-block,0.4,random_forest,7,126,050dbe4757d441557c6c763722fd3bde1f6852d365e546...,14745.446463,6960.346310,12216.633596,...,6149.317460,11895.615076,0.713926,3516.593600,1721.876355,5238.469955,0.357143,6960.346310,8682.222665,12198.816264


### Understanding the selected-model diagnostics

Each row describes the selected random forest on one validation scenario. `counties` is the size of the validation split; `targets` counts the hidden cells actually scored and can cover fewer than seven counties. `mask_sha256` identifies the mask, and `cv_mae` is the grouped training-fold score.

- **MAE** is the average size of the errors, ignoring direction; here it ranges from about **2,953 to 7,754 persons per hidden cell**.
- **RMSE** gives larger errors more influence; it ranges from about **4,903 to 12,217**. **R²** compares squared errors with predicting the evaluation sample's mean: closer to 1 is better, and a negative value can occur.
- The `median_baseline` columns score a simple reference that predicts each age-sex value using only the training counties' median. Its MAE is lower in **three of nine scenarios**: age-block 10% and 20%, and county-block 40%.
- `bias` and the cost columns describe error direction and assumed costs, explained below.

These are validation results, not final test results or proof that the model will perform similarly in other years or counties.

In [6]:
model_charts = model_figures(outputs)
model_charts['validation'].show()

### Understanding the validation comparison chart

The horizontal axis shows the mask pattern, the vertical axis shows validation **MAE**, and colors distinguish model families. The separate panels show mask rates. Compare bars within the **same pattern and rate**: a shorter bar means smaller average error on those hidden validation values.

Random forest has the best average across all nine scenarios, but the shorter bars vary by scenario. Repeatedly adjusting models based on the same small validation set can make the apparent winner look more reliable than it is.

In [6]:
model_charts['generalisation'].show()

### Understanding the cross-validation chart

Each scenario has two bars: `cv_mae` is the grouped cross-validation error from training counties, and `mae` is the error on separate validation counties. Lower bars mean lower average absolute error. They are not a training-loss curve because they come from different groups of counties.

For county-block 10%, CV MAE is about **26,986**, compared with about **7,754** on validation. Only three training counties have masked targets in this case, making the CV estimate especially unstable. Differences in the samples can explain large gaps; the chart alone cannot prove or rule out overfitting.

The median baseline predicts each age-sex cell using only training-county medians. Compare its MAE and RMSE with the selected model for each scenario. A large validation-versus-CV gap warrants investigation, but these estimates use different small county samples and are not an in-sample training loss curve. County-block 10% has only three eligible training counties. Selection on seven validation counties adds optimism; no procedure here can guarantee absence of overfitting or good future performance. Consistency columns compare against potentially inconsistent reported totals. The autoencoder also receives aggregate training supervision unavailable to the trees.

## Cost of wrong predictions
Underestimation can lead to insufficient planned resources, while overestimation can lead to over-allocation. Their true costs depend on the decision and are unknown. The following sensitivity analysis assigns illustrative costs per person of (under, over) = (1, 1), (2, 1), and (1, 2).

In [7]:
display(outputs['diagnostics'][['pattern', 'rate', 'targets', 'bias',
    'mean_underestimate', 'mean_overestimate', 'underestimate_fraction',
    'cost_equal', 'cost_under_double', 'cost_over_double']])

,pattern,rate,targets,bias,mean_underestimate,mean_overestimate,underestimate_fraction,cost_equal,cost_under_double,cost_over_double
0,random,0.1,29,-209.262759,1580.909655,1371.646897,0.551724,2952.556552,4533.466207,4324.203448
1,random,0.2,59,-2126.377458,3317.772373,1191.394915,0.440678,4509.167288,7826.939661,5700.562203
2,random,0.4,118,-2867.553220,3546.798475,679.245254,0.567797,4226.043729,7772.842203,4905.288983
3,age-block,0.1,30,352.298667,3325.439333,3677.738000,0.366667,7003.177333,10328.616667,10680.915333
4,age-block,0.2,58,-2347.285690,3316.672414,969.386724,0.586207,4286.059138,7602.731552,5255.445862
5,age-block,0.4,118,-4242.602034,4899.308814,656.706780,0.627119,5556.015593,10455.324407,6212.722373
6,county-block,0.1,42,-4454.063707,6103.824529,1649.760822,0.404762,7753.585350,13857.409879,9403.346172
7,county-block,0.2,42,6813.484366,90.202208,6903.686574,0.071429,6993.888781,7084.090989,13897.575355
8,county-block,0.4,126,3516.593600,1721.876355,5238.469955,0.357143,6960.346310,8682.222665,12198.816264


### Understanding error direction and assumed costs

The table studies what happens when predictions are too low or too high:

- `bias` is the average of **prediction minus actual**. A negative value indicates net underprediction; a positive value indicates net overprediction.
- `mean_underestimate` and `mean_overestimate` are the error amounts averaged across **all** hidden cells, including zero for cells with no error in that direction. `underestimate_fraction` is the share of hidden cells predicted below the true value.
- `cost_equal` weights under- and overestimation equally and equals MAE. `cost_under_double` doubles the weight of underestimates; `cost_over_double` doubles the weight of overestimates.

For random 20%, bias is about **-2,126 persons per cell**. Equal-weight loss is about **4,509**, rising to **7,827** when underestimation is given twice the weight. These weights are illustrative, not measured prices or financial losses.

In [7]:
model_charts['costs'].show()

### Understanding the cost sensitivity chart

The horizontal axis lists scenarios; the vertical axis shows **average assumed loss per hidden cell**. The colors compare equal error weights, double weight for underestimation, and double weight for overestimation. A higher bar means a higher loss under that assumption.

For county-block 20%, the overestimation-weighted loss is about **13,898**, versus about **7,084** with double weight on underestimation. This reflects the model's error direction in that scenario, not an actual cost in currency. The chart does not prove which model would be best for a real allocation decision.

Costs are average hypothetical units per artificially masked cell, not currency, actual financial loss or a validated downstream decision outcome. Positive bias means overprediction. Equal-cost loss equals MAE; asymmetric losses increase when the corresponding error direction is more expensive. The MAE-selected model is not necessarily optimal for either asymmetric cost.

## Simulated prioritisation
The optimisation uses observed profiles rather than reconstructed predictions. Each selected county has one simulated cost unit, with child coverage, elderly coverage and a dependency-burden proxy as objectives.

In [8]:
display(tables['indicators'])

,admin1_code,admin1_name,children,working_age,elderly,total,child_share,elderly_share,dependency_ratio
0,KE001,Mombasa,393313.0,792072.0,22892.0,1208277.0,0.325516,0.018946,0.525464
1,KE002,Kwale,387321.0,446434.0,33035.0,866790.0,0.446845,0.038112,0.941586
2,KE003,Kilifi,613788.0,784069.0,55888.0,1453745.0,0.422212,0.038444,0.854103
3,KE004,Tana River,152837.0,153294.0,9804.0,315935.0,0.483761,0.031032,1.060974
4,KE005,Lamu,55957.0,82250.0,5705.0,143912.0,0.388828,0.039642,0.749690
5,KE006,Taita Taveta,115882.0,205050.0,19729.0,340661.0,0.340168,0.057914,0.661356
6,KE007,Garissa,383495.0,443437.0,14382.0,841314.0,0.455829,0.017095,0.897257
7,KE008,Wajir,392639.0,376933.0,11640.0,781212.0,0.502602,0.014900,1.072549
8,KE009,Mandera,475723.0,379079.0,12615.0,867417.0,0.548436,0.014543,1.288222
9,KE010,Marsabit,215807.0,230067.0,13887.0,459761.0,0.469390,0.030205,0.998379


### Understanding the county indicators

Each row is a county, using **observed** age-sex counts rather than model predictions. `children` covers ages 0 to 14; `working_age`, ages 15 to 64; and `elderly`, ages 65 and above. `total` is their sum.

`child_share` and `elderly_share` divide the respective counts by that total. `dependency_ratio` divides children plus elderly by the working-age count. For Mombasa, the calculated total is **1,208,277**, while the source reports **1,208,303**; this illustrates the consistency issue already identified.

In [8]:
display(outputs['optimisation'])

{'profile_source': 'observed age-sex population',
 'simulated_budget_units': 10,
 'cost_per_county_units': 1,
 'seed': 42,
 'generations': 200,
 'weighted_sum_weight_vectors': 66,
 'hypervolume_reference_minimization': [0.01, 0.01, 0.01],
 'hypervolume': {'nsga3': 0.04057937731659412,
  'weighted_sum': 0.04208357198495907},
 'front_sizes': {'nsga3': 21, 'weighted_sum': 28},
 'input_sha256': 'a040134935fb992e176c44163e78565bff40518f5e7cd0d6e0ec91b9c2d1ffd1'}

### Understanding the optimisation summary

The simulation selects counties from observed profiles. Its budget is **10 units**, at **one unit per county**; it is not a monetary budget. The weighted-sum method explored **66** weight combinations, while NSGA-III used **200 generations** and seed 42.

The output contains **28** nondominated weighted-sum solutions and **21** NSGA-III solutions. Their reported hypervolumes are approximately **0.04208** and **0.04058**. Hypervolume summarizes the coverage of the objective space relative to a chosen reference point; higher is better under that particular setup, but results depend on that reference, the weight grid and search settings. They do not prove which method is best in practice.

In [8]:
display(outputs['tradeoffs'])

,method,selected_count,selected_counties,child_coverage,elderly_coverage,dependency_burden_coverage
0,weighted_sum,10,"[""KE012"", ""KE015"", ""KE016"", ""KE021"", ""KE022"", ...",0.357366,0.401344,0.187812
1,weighted_sum,10,"[""KE012"", ""KE015"", ""KE016"", ""KE021"", ""KE022"", ...",0.312479,0.396670,0.200635
2,weighted_sum,10,"[""KE012"", ""KE015"", ""KE016"", ""KE017"", ""KE021"", ...",0.353783,0.405221,0.182959
3,weighted_sum,10,"[""KE012"", ""KE015"", ""KE016"", ""KE017"", ""KE021"", ...",0.303863,0.405925,0.194277
4,weighted_sum,10,"[""KE012"", ""KE015"", ""KE016"", ""KE017"", ""KE019"", ...",0.293834,0.406461,0.186533
5,weighted_sum,10,"[""KE009"", ""KE015"", ""KE021"", ""KE024"", ""KE033"", ...",0.275044,0.273102,0.251687
6,weighted_sum,10,"[""KE008"", ""KE009"", ""KE024"", ""KE032"", ""KE033"", ...",0.347688,0.231656,0.246806
7,weighted_sum,10,"[""KE004"", ""KE008"", ""KE009"", ""KE024"", ""KE025"", ...",0.248087,0.171481,0.272356
8,weighted_sum,10,"[""KE003"", ""KE012"", ""KE022"", ""KE032"", ""KE033"", ...",0.386644,0.335673,0.204735
9,weighted_sum,10,"[""KE003"", ""KE012"", ""KE022"", ""KE032"", ""KE033"", ...",0.386170,0.342766,0.204455


### Understanding the candidate allocations

Each row is a different simulated selection. `method` identifies how it was found, `selected_count` is the number of counties, and `selected_counties` lists their county codes. All **49 displayed solutions** select 10 counties.

The remaining columns show the selected counties' shares of the total child population, elderly population and summed dependency ratios. For instance, a value of 0.30 represents a 30% share of the corresponding objective. Improving one share can reduce another; the table compares trade-offs and does not specify actual funding or services.

In [8]:
model_charts['tradeoffs'].show()

### Understanding the three-dimensional trade-off plot

Each point is a candidate county selection. The three axes show the selected share of child population, elderly population and summed county dependency ratios; colors identify the optimisation method. Larger values on an axis mean a larger share of that particular objective under the simulation.

Different points emphasize different objectives, so there is no universally best point without a decision maker's priorities. The simulation assumes selecting a county covers its entire indicated population; the dependency objective is a proxy, not a validated measure of fairness or service delivery.

Inspect trade-offs rather than treating a single solution as universally best. NSGA-III is approximate, and hypervolume depends on its reference point. The burden proxy is not validated fairness or actual service coverage; these simulated costs are not the prediction-error costs above.

## Optional final held-out test
Only after the methodology and selection are frozen, set `AML_EVALUATE_TEST=1` before starting the notebook to evaluate the selected family on the seven reserved test counties. Do not repeatedly tune decisions using these scores. The same evaluation is available through `python -m src.models.diagnostics --split test`.

In [9]:
if os.environ.get('AML_EVALUATE_TEST') == '1':
    test_path = run_dir / 'reports' / 'selected_test_diagnostics.csv'
    if test_path.exists():
        raise FileExistsError(f'Test report already exists: {test_path}')
    test_results = evaluate_selected(run_dir / 'data' / 'kenya_county_population.csv',
        run_dir / 'models' / 'selected', split_name='test')
    test_results.to_csv(test_path, index=False)
    display(test_results)
else:
    print('Test evaluation skipped; held-out counties remain reserved.')

Test evaluation skipped; held-out counties remain reserved.


### Understanding the held-out test output

In the saved run, the message says **test evaluation was skipped**. Seven counties remain reserved for a final assessment of the selected model after the methodology is frozen; listing test masks earlier did not score them.

If the optional test cell is enabled in a later run, its table reports the same error, baseline and illustrative cost measures on the held-out cells. A test report from only seven counties remains uncertain and should not be repeatedly used to tune the model.

If enabled, this is a final estimate from a very small, simulated-mask test set, not proof of deployment readiness or protection against distribution shift. Preserve the report and disclose the sample size, uncertainty, source inconsistencies and all modeling assumptions.